# Week 8 | Narrative dashboard
**Goal:** build Context → Conflict → Insight → Action using Streamlit and Plotly.

Launch from the project terminal:
```powershell
.\.venv\Scripts\python.exe -m streamlit run Source_Code/dashboard.py
```
This opens a local interactive dashboard. It does not require a cloud account.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image
ROOT = next((p if (p/'Source_Code'/'attrition_lab.py').exists() else p/'DSV_PROJECT'
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p/'Source_Code'/'attrition_lab.py').exists() or (p/'DSV_PROJECT'/'Source_Code'/'attrition_lab.py').exists()), None)
if ROOT is None: raise FileNotFoundError('Open this notebook inside DSV_PROJECT')
sys.path.insert(0,str(ROOT/'Source_Code'))
from attrition_lab import load_raw, clean_data, rate_table
raw=load_raw(); clean=clean_data(raw)
pd.set_option('display.max_columns',12)

In [2]:
chapters=pd.DataFrame([
    ['Overview','Context','Dataset composition and headline questions'],
    ['Data audit','Conflict','Missingness, outliers and transformations'],
    ['Explore groups','Explore','Role/age/scenario filters; group intervals and distributions'],
    ['Evidence checks','Resolution','Full-cohort uncertainty, five scenarios and role consistency'],
    ['PCA map','Structure','Variance, 2D/3D projection and component coefficients'],
    ['Next actions','Action','Source clarification and prospective validation']
],columns=['chapter','story_role','content'])
display(chapters)
print('Interactive plot types: bar, histogram, box, violin, scatter, heatmap, line, 3D scatter')

,chapter,story_role,content
0,Overview,Context,Dataset composition and headline questions
1,Data audit,Conflict,"Missingness, outliers and transformations"
2,Explore groups,Explore,Role/age/scenario filters; group intervals and...
3,Evidence checks,Resolution,"Full-cohort uncertainty, five scenarios and ro..."
4,PCA map,Structure,"Variance, 2D/3D projection and component coeff..."
5,Next actions,Action,Source clarification and prospective validation


Interactive plot types: bar, histogram, box, violin, scatter, heatmap, line, 3D scatter


## Interaction semantics
Explore-group filters recalculate that cohort's denominator, share and intervals. Cleaning
scenarios are constructed on the full raw file before display filters, preserving comparable
imputation rules. Empty cohorts show a clear message.

Evidence checks always display the full-cohort scorecard and say so. PCA is fitted once on
the full baseline; filters change displayed records, not the PCA basis. Large scatterplots use
fixed samples with displayed sample sizes. No employee ID appears in tooltips.

CSV downloads contain aggregated group/evidence tables.

In [3]:
from streamlit.testing.v1 import AppTest
app=AppTest.from_file(str(ROOT/'Source_Code'/'dashboard.py'),default_timeout=60).run()
assert len(app.exception)==0
print('Overview rendered successfully.')
app.radio(key='page').set_value('Explore groups').run()
assert len(app.exception)==0
print('Default cohort records:',app.metric[0].value)
app.multiselect(key='roles').set_value([]).run()
assert len(app.exception)==0
print('Empty cohort message:',app.warning[0].value)

2026-09-25 09:28:25.359 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Overview rendered successfully.


Default cohort records: 74,498
Empty cohort message: No records match these filters. Select at least one role or widen the age range.


**Demo sequence:** Overview → Data audit → Explore groups → Evidence checks → PCA map → Next actions.
Show at least five plot types. The app provides eight types, including an optional violin and 3D view.

[Streamlit testing documentation](https://docs.streamlit.io/develop/api-reference/app-testing)